In [1]:
import Pkg
Pkg.activate("../..")

  Activating project at `~/.julia/dev/DiffusionEvolution`


In [2]:
using Revise

In [3]:
using DiffusionEvolution, PyPlot, LinearAlgebra, Statistics, JLD2

Precompiling DiffusionEvolution
  ✓ DiffusionEvolution
  1 dependency successfully precompiled in 2 seconds. 132 already precompiled.


In [4]:
const de = DiffusionEvolution

DiffusionEvolution

In [ ]:
import PyPlot.subplots
subplots(x, y, d) = subplots(x, y, figsize=(y*d, x*d))

In [ ]:
d = 10
nsamples = 1000
T = 50

In [ ]:
M = randn(d,d) * 0.5
J = M * M'

In [ ]:
eigen(inv(J))

In [ ]:
matshow(inv(J))
colorbar()

In [ ]:
θ0 = zeros(d)
θ1 = 1.5 * randn(d)
θ = θ0 .+ (θ1 .* randn(d))
θ |> norm

In [ ]:
γ = 0.1

In [ ]:
x = simulate_ou_process(J, θ, γ, nsamples=nsamples, T=T);

In [ ]:
tpoints = [1,2,3,4,5,6,7,8,9,10,20,30,40,50]
fig, ax = subplots(1, length(tpoints), 4)
for t in eachindex(tpoints)
    ax[t].hist2d(x[1,:,tpoints[t]], x[2,:,tpoints[t]], bins=20)
    ax[t].scatter(θ[1], θ[2], marker="o", color="red")
    ax[t].set_title("time t=$(tpoints[t])")
end

In [ ]:
fig, ax = subplots(1,3,5)
μ_eq = mean(x[:,:,end], dims=2)
Σ_eq = ((x[:,:,end] .- μ_eq)*(x[:,:,end] .- μ_eq)')./nsamples

ax[1].scatter(θ, μ_eq)
ax[1].set_xlabel("teacher θ")
ax[1].set_ylabel("empirical average (last round)")
rho_theta = cor(vec(θ), vec(μ_eq))
ax[1].set_title("average, pearson=$(round(rho_theta, digits=2))")


ax[2].scatter(vec(Σ_eq), vec(inv(J)))
ax[2].set_xlabel("empirical covariance (last round)")
ax[2].set_ylabel("teacher inverse J")
rho_sigma = cor(vec(Σ_eq), vec(inv(J)))
ax[2].set_title("covariance, pearson=$(round(rho_sigma, digits=2))")

ax[3].scatter(vec(inv(Σ_eq)), vec(J))
ax[3].set_xlabel("empirical inverse covariance")
ax[3].set_ylabel("teacher J")
rho_J = cor(vec(inv(Σ_eq)), vec(J))
ax[3].set_title("covariance, pearson=$(round(rho_J, digits=2))")

In [ ]:
t_sample=[1, 2, 3, 4, 5, 6]

In [ ]:
fig, ax = subplots(1,3,5)
μ_last = mean(x[:,:,t_sample[end]], dims=2)
Σ_last = ((x[:,:,t_sample[end]] .- μ_last)*(x[:,:,t_sample[end]] .- μ_last)')./nsamples

ax[1].scatter(θ, μ_last)
ax[1].set_xlabel("teacher θ")
ax[1].set_ylabel("empirical average (last round)")
rho_theta = cor(vec(θ), vec(μ_last))
ax[1].set_title("average, pearson=$(round(rho_theta, digits=2))")


ax[2].scatter(vec(Σ_last), vec(inv(J)))
ax[2].set_xlabel("empirical covariance (last round)")
ax[2].set_ylabel("teacher inverse J")
rho_sigma = cor(vec(Σ_last), vec(inv(J)))
ax[2].set_title("covariance, pearson=$(round(rho_sigma, digits=2))")

ax[3].scatter(vec(inv(Σ_last)), vec(J))
ax[3].set_xlabel("empirical inverse covariance")
ax[3].set_ylabel("teacher J")
rho_J = cor(vec(inv(Σ_last)), vec(J))
ax[3].set_title("covariance, pearson=$(round(rho_J, digits=2))")

In [ ]:
counts = ones(nsamples, length(t_sample)) ./ nsamples
deltas = [1,1,1,1,1]
data = collect_data(x[:,:,t_sample], counts, deltas);

In [ ]:
l_values_gamma, x_opt_gamma, status_gamma = learn_gamma_nlopt(data, initialize=length(t_sample), λ=0.0, prior=0.01, ftol_rel=1e-9)

In [ ]:
@save "test3_A_save.jld2" J θ x l_values_gamma x_opt_gamma status_gamma

In [ ]:
x_opt_gamma[end]

In [ ]:
J_inferred = de.compute_J(x_opt_gamma, data.d);
θ_inferred = de.compute_theta(x_opt_gamma, data.d);

In [ ]:
fig, ax = subplots(1, 4, 6)
ax[1].scatter(J, J_inferred)
rho_J = cor(vec(J), vec(J_inferred))
ax[1].set_xlabel("J teacher")
ax[1].set_ylabel("J student")
ax[1].set_title("pearson=$(round(rho_J, digits=2))")

ax[2].scatter(inv(J), inv(J_inferred))
rho_invJ = cor(vec(inv(J)), vec(inv(J_inferred)))
ax[2].set_xlabel("inverse J teacher")
ax[2].set_ylabel("inverse J student")
ax[2].set_title("pearson=$(round(rho_invJ, digits=2))")

ax[3].scatter(Σ_eq, inv(J_inferred))
rho_sigma_invJ = cor(vec(Σ_eq), vec(inv(J_inferred)))
ax[3].set_xlabel("Σ empirical")
ax[3].set_ylabel("inverse J student")
ax[3].set_title("pearson=$(round(rho_sigma_invJ, digits=2))")

ax[4].scatter(θ, θ_inferred)
rho_theta = cor(vec(θ), θ_inferred)
ax[4].set_xlabel("θ")
ax[4].set_ylabel("inferred θ")
ax[4].set_title("pearson=$(round(rho_theta, digits=2))")

In [ ]:
γ = collect(0.001:0.002:1.0)#vcat([0.001, 0.005], collect(0.1:0.1:1.5))

In [ ]:
lγ = map(x->de.log_likelihood(x_opt_gamma[1:end-1], data, x, 0.0, 0.01), γ)

In [ ]:
plot(γ, lγ .+ (minimum(lγ) + 1.0), linestyle="dashed", marker="o")
xscale(:log)